# MitoGuard — Google Colab setup and execution

This notebook provides a reproducible Colab entry point for repository setup, persistent storage, dataset wiring, integrity gates, the current candidate-level pilot, and artifact export.

**Important:** the repository currently contains the validated CPU pilot, not the proposed YOLO + DINOv3 research training implementation. This notebook will not present pilot results as full-model results. The 65 GB Track 1 download is disabled by default.

## 1. Runtime configuration
Select a GPU runtime in Colab: **Runtime → Change runtime type → GPU**. Edit only the values below before using **Run all**.

In [ ]:
REPO_URL = "https://github.com/Ajogeorge29/MIDOG_2025.git"
BRANCH = "main"
COLAB_REPO = "/content/MIDOG_2025"
DRIVE_ROOT = "/content/drive/MyDrive/MitoGuard"

MOUNT_DRIVE = True             # persistent datasets, reports, and future checkpoints
DOWNLOAD_TRACK2 = True         # approximately 426 MB
DOWNLOAD_TRACK1_METADATA = True
DOWNLOAD_TRACK1_IMAGES = False # approximately 65 GB; deliberately opt-in
RUN_AUDIT = True
RUN_PILOT = True               # requires Track 1 TIFFs as well as Track 2
AUDIT_WORKERS = 4
TRACK1_DOWNLOAD_WORKERS = 4
SEED = 2025

## 2. GPU, memory, and disk preflight

In [ ]:
import os, platform, shutil, subprocess, sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=True)
else:
    print("WARNING: no NVIDIA GPU is attached. Setup/audit can run, but deep training should not.")
total, used, free = shutil.disk_usage("/content")
print(f"Local disk: {free / 2**30:.1f} GiB free of {total / 2**30:.1f} GiB")

## 3. Mount Drive and clone/update the repository

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

repo = Path(COLAB_REPO)
if (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
os.chdir(repo)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository:", repo)
print("Commit:", commit)

## 4. Install and verify the environment

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
import numpy, pandas, sklearn, skimage, torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
subprocess.run([sys.executable, "-m", "pytest"], check=True)

## 5. Persistent data and report locations
Datasets remain in Drive across Colab sessions. The environment variables are set before importing repository modules.

In [ ]:
storage_root = Path(DRIVE_ROOT if MOUNT_DRIVE else "/content/MitoGuard")
data_root = storage_root / "data" / "raw"
report_root = storage_root / "reports" / commit[:12]
model_root = storage_root / "models" / commit[:12]
for path in (data_root, report_root, model_root):
    path.mkdir(parents=True, exist_ok=True)
os.environ["MIDOG_DATA_ROOT"] = str(data_root)
os.environ["MIDOG_REPORT_ROOT"] = str(report_root)
os.environ["PYTHONHASHSEED"] = str(SEED)
print("Data:", data_root)
print("Reports:", report_root)
print("Models:", model_root)

## 6. Download/wire datasets
Track 2 and Track 1 metadata are safe defaults. Only enable the full Track 1 download after confirming Drive quota, runtime duration, and dataset terms. Downloads are resumable.

In [ ]:
def run_midog(*args):
    command = [sys.executable, "-m", "midog2025.download", "--data-root", str(data_root), *args]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

if DOWNLOAD_TRACK2:
    run_midog("--track", "track2")
if DOWNLOAD_TRACK1_METADATA:
    run_midog("--track", "track1-metadata")
if DOWNLOAD_TRACK1_IMAGES:
    total, used, free = shutil.disk_usage(storage_root)
    if free < 75 * 2**30:
        raise RuntimeError(f"Track 1 needs a safety margin of 75 GiB; only {free / 2**30:.1f} GiB is free")
    run_midog("--track", "track1-images", "--i-understand-size", "--workers", str(TRACK1_DOWNLOAD_WORKERS))

## 7. Data audit and leakage gate

In [ ]:
if RUN_AUDIT:
    command = [sys.executable, "-m", "midog2025.audit", "--data-root", str(data_root),
               "--report-root", str(report_root / "audit"), "--workers", str(AUDIT_WORKERS)]
    subprocess.run(command, check=True)
    from IPython.display import display, HTML
    summary_path = report_root / "audit" / "audit_summary.json"
    if summary_path.exists():
        import json
        display(json.loads(summary_path.read_text()))

## 8. Candidate-level end-to-end pilot
This is an engineering sanity check using HOG/color features and SGD classifiers. It requires the full Track 1 TIFF set. It is not YOLO/DINOv3 training and not a challenge-performance estimate.

In [ ]:
track1_images = data_root / "track1" / "images"
tiff_count = len(list(track1_images.glob("*.tiff"))) if track1_images.exists() else 0
if RUN_PILOT and tiff_count:
    pilot_output = report_root / "pilot"
    command = [sys.executable, "-m", "midog2025.pilot", "--data-root", str(data_root),
               "--output", str(pilot_output)]
    subprocess.run(command, check=True)
    from IPython.display import Image as DisplayImage, display
    display(DisplayImage(filename=str(pilot_output / "pilot_confusion_matrix.png")))
elif RUN_PILOT:
    print("Pilot skipped: no Track 1 TIFFs found. Upload/sync them to", track1_images,
          "or explicitly enable DOWNLOAD_TRACK1_IMAGES.")

## 9. Main GPU training readiness gate
The proposed full system consists of a high-recall YOLO-family proposal detector, a LoRA-adapted DINOv3 local/context encoder, a verification head, a subtype head, residual context fusion, and post-hoc calibration. Those training modules are not yet implemented in this repository. The following cell records readiness without fabricating a training result.

In [ ]:
readiness = {
    "commit": commit,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "track1_tiff_count": tiff_count,
    "track2_csv_present": (data_root / "track2" / "MIDOG25_Atypical_Classification_Train_Set.csv").exists(),
    "deep_training_code_present": (repo / "src" / "midog2025" / "training").exists(),
}
import json, datetime
readiness["created_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
readiness_path = report_root / "training_readiness.json"
readiness_path.write_text(json.dumps(readiness, indent=2), encoding="utf-8")
print(json.dumps(readiness, indent=2))
if not readiness["deep_training_code_present"]:
    print("STOP: setup is ready, but the proposed deep-model trainer must be implemented and tested before main training.")

## 10. Package the run manifest
Reports remain in Drive. This final cell also creates a small downloadable archive without copying the raw datasets.

In [ ]:
archive_base = Path("/content") / f"mitoguard_run_{commit[:12]}"
archive = shutil.make_archive(str(archive_base), "zip", root_dir=report_root)
print("Run reports:", report_root)
print("Downloadable archive:", archive)
try:
    from google.colab import files
    print("Use files.download(archive) if you want to download it now.")
except ImportError:
    pass